# rsna knee s6ft trainer

warm-start fine-tune of nartaa's all-data s6 convnext (s6fall, cc by-nc 4.0) on multi-teacher targets (report labels + out-of-fold image models). private training run on a cache of non-gold training studies; no test data, no gold studies.

In [ ]:
# locate the inputs, then train on every gpu (one process per gpu)
import glob, os, subprocess, sys, torch
def one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, pattern
    return hits[0]
code = os.path.dirname(one('/kaggle/input/**/s6ft_train.py'))
init = one('/kaggle/input/**/E_prior_s42_fall.pt')
targets = one('/kaggle/input/**/training_labels_v4_superior.csv')
cache = os.path.dirname(code)
n_gpu = torch.cuda.device_count()
print('gpus', n_gpu, '| code', code, '| init', init, flush=True)
subprocess.run(['nvidia-smi'], check=False)
EPOCHS = '3'
cmd = ([sys.executable, '-m', 'torch.distributed.run', f'--nproc_per_node={n_gpu}'] if n_gpu > 1 else [sys.executable]) + [
    f'{code}/s6ft_train.py', '--cache', cache, '--targets', targets, '--init', init, '--code', code,
    '--out', '/kaggle/working/s6ft', '--epochs', EPOCHS, '--workers', '2', '--max-hours', '10.5']
rc = subprocess.run(cmd).returncode
print('trainer exit code', rc, flush=True)
assert rc == 0 and os.path.isfile('/kaggle/working/s6ft/s6ft_ema.pt')
